In [ ]:
import lumapi
import numpy as np
import matplotlib.pyplot as plt
fdtd = lumapi.FDTD()
# ========================
# Constants
# ========================
N_CORE   = np.sqrt(12.0)
N_BG     = 1.0
LAMBDA0  = 1.55e-6
DX       = 20e-9

GRID_X_LENGTH = 6.3e-6
GRID_Y_LENGTH = 12e-6
GRID_Z_LENGTH = 1.5e-6

BUS_LENGTH   = 2.0e-6
BUS_WIDTH    = 0.2e-6
BUS_CENTER_X = 1.0e-6
BUS_CENTER_Y = 0.0

WEDGE_ARM_LENGTH_X  = 0.5e-6
WEDGE_RECT_LENGTH_X = 0.5e-6

DESIGN_LENGTH_X = 1.0e-6
DESIGN_SPAN_Y   = 10.0e-6
DUMMY_LENGTH_Z  = 0.5e-6

NUM_OUTPUT_WG      = 2
OUTPUT_WG_LENGTH_X = 2.4e-6
OUTPUT_WG_WIDTH    = 0.2e-6

# source / monitor settings
SRC_Y_SPAN = 0.4e-6
SRC_Z_SPAN = 1.0e-6
MON_Y_SPAN = 0.4e-6
MON_Z_SPAN = 1.0e-6


def build_sim(fdtd, output_wg_gap):
    """
    Build one FDTD simulation for a given output waveguide gap.
    """

    fdtd.newproject()
    fdtd.addfdtd()

    # ========================
    # FDTD region
    # ========================

    fdtd.set("dimension", "3D")
    fdtd.set("x span", GRID_X_LENGTH)
    fdtd.set("y span", GRID_Y_LENGTH)
    fdtd.set("z span", GRID_Z_LENGTH)

    fdtd.set("x", GRID_X_LENGTH / 2)
    fdtd.set("y", 0.0)
    fdtd.set("z", 0.0)
    fdtd.set("index", N_BG)

    fdtd.set("x min bc", "PML")
    fdtd.set("x max bc", "PML")
    fdtd.set("y min bc", "PML")
    fdtd.set("y max bc", "PML")
    fdtd.set("z min bc", "PML")
    fdtd.set("z max bc", "PML")
    fdtd.set("pml layers", 10)

    # ========================
    # Mesh
    # ========================
    fdtd.addmesh()
    fdtd.set("name", "mesh_global")
    fdtd.set("x", GRID_X_LENGTH / 2)
    fdtd.set("y", 0.0)
    fdtd.set("z", 0.0)
    fdtd.set("x span", GRID_X_LENGTH)
    fdtd.set("y span", GRID_Y_LENGTH)
    fdtd.set("z span", GRID_Z_LENGTH)
    fdtd.set("dx", DX)
    fdtd.set("dy", DX)
    fdtd.set("dz", DX)

    # ========================
    # 1) Input bus
    # ========================
    fdtd.addrect()
    fdtd.set("name", "bus")
    fdtd.set("x", BUS_CENTER_X)
    fdtd.set("y", BUS_CENTER_Y)
    fdtd.set("z", 0.0)
    fdtd.set("x span", BUS_LENGTH)
    fdtd.set("y span", BUS_WIDTH)
    fdtd.set("z span", DUMMY_LENGTH_Z)
    fdtd.set("index", N_CORE)

    bus_right_edge_x = BUS_CENTER_X + BUS_LENGTH / 2

    # ========================
    # 2) Wedge / taper
    # ========================
    taper_x0 = bus_right_edge_x
    taper_x1 = bus_right_edge_x + WEDGE_ARM_LENGTH_X

    w0 = BUS_WIDTH
    w1 = DESIGN_SPAN_Y
    y0 = BUS_CENTER_Y

    fdtd.addpoly()
    fdtd.set("name", "wedge_taper")
    fdtd.set("x", 0.0)
    fdtd.set("y", 0.0)
    fdtd.set("z", 0.0)
    fdtd.set("z span", DUMMY_LENGTH_Z)
    fdtd.set("index", N_CORE)

    verts = np.array([
        [taper_x0, y0 - w0 / 2],
        [taper_x0, y0 + w0 / 2],
        [taper_x1, y0 + w1 / 2],
        [taper_x1, y0 - w1 / 2],
    ])
    fdtd.set("vertices", verts)

    # ========================
    # 3) Rectangle after taper
    # ========================
    rect_x0 = taper_x1
    rect_center_x = rect_x0 + WEDGE_RECT_LENGTH_X / 2

    fdtd.addrect()
    fdtd.set("name", "rectangle1")
    fdtd.set("x", rect_center_x)
    fdtd.set("y", y0)
    fdtd.set("z", 0.0)
    fdtd.set("x span", WEDGE_RECT_LENGTH_X)
    fdtd.set("y span", DESIGN_SPAN_Y)
    fdtd.set("z span", DUMMY_LENGTH_Z)
    fdtd.set("index", N_CORE)

    # ========================
    # 4) Design region
    # ========================
    design_left_edge_x = rect_x0 + WEDGE_RECT_LENGTH_X
    design_right_edge_x = design_left_edge_x + DESIGN_LENGTH_X
    design_center_x = 0.5 * (design_left_edge_x + design_right_edge_x)

    fdtd.addrect()
    fdtd.set("name", "design_spatial")
    fdtd.set("x", design_center_x)
    fdtd.set("y", y0)
    fdtd.set("z", 0.0)
    fdtd.set("x span", DESIGN_LENGTH_X)
    fdtd.set("y span", DESIGN_SPAN_Y)
    fdtd.set("z span", DUMMY_LENGTH_Z)
    fdtd.set("index", N_CORE)

    # ========================
    # 5) Output waveguides
    # ========================
    output_wg_center_spacing = OUTPUT_WG_WIDTH + output_wg_gap
    output_left_edge_x = design_right_edge_x
    output_center_x = output_left_edge_x + OUTPUT_WG_LENGTH_X / 2

    mid = (NUM_OUTPUT_WG - 1) / 2.0
    output_y_list = []

    for i in range(NUM_OUTPUT_WG):
        y_c = (i - mid) * output_wg_center_spacing
        output_y_list.append(y_c)

        fdtd.addrect()
        fdtd.set("name", f"out_wg_{i:02d}")
        fdtd.set("x", output_center_x)
        fdtd.set("y", y_c)
        fdtd.set("z", 0.0)
        fdtd.set("x span", OUTPUT_WG_LENGTH_X)
        fdtd.set("y span", OUTPUT_WG_WIDTH)
        fdtd.set("z span", DUMMY_LENGTH_Z)
        fdtd.set("index", N_CORE)

    # ========================
    # 6) Source
    # ========================
    fdtd.addmode()
    fdtd.set("name", "src_bus")
    fdtd.set("injection axis", "x-axis")
    fdtd.set("direction", "Forward")
    fdtd.set("x", 0.3e-6)
    fdtd.set("y", BUS_CENTER_Y)
    fdtd.set("z", 0.0)
    fdtd.set("y span", SRC_Y_SPAN)
    fdtd.set("z span", SRC_Z_SPAN)
    fdtd.set("center wavelength", LAMBDA0)
    fdtd.set("wavelength span", 0)

    # make source solve and use fundamental mode
    fdtd.updatesourcemode(1)

    # ========================
    # 7) Output monitors
    # ========================
    output_right_edge_x = output_left_edge_x + OUTPUT_WG_LENGTH_X - 0.4e-6
    x_mon = output_right_edge_x

    for i, y_c in enumerate(output_y_list):
        fdtd.adddftmonitor()
        fdtd.set("name", f"mon_out_{i:02d}")
        fdtd.set("monitor type", "2D X-normal")
        fdtd.set("x", x_mon)
        fdtd.set("y", y_c)
        fdtd.set("z", 0.0)
        fdtd.set("y span", MON_Y_SPAN)
        fdtd.set("z span", MON_Z_SPAN)

    return output_y_list


def run_single_gap(output_wg_gap, save_fsp=False):
    """
    Build, run, and return transmissions for one gap value.
    """
    with lumapi.FDTD(hide=True) as fdtd:
        build_sim(fdtd, output_wg_gap)

        if save_fsp:
            fdtd.save(f"gap_{output_wg_gap*1e6:.3f}um.fsp")

        fdtd.run()

        # For one frequency point, transmission(...) returns a scalar or length-1 array.
        T_list = []
        for i in range(NUM_OUTPUT_WG):
            mon_name = f"mon_out_{i:02d}"
            T_i = fdtd.transmission(mon_name)
            T_i = np.asarray(T_i).squeeze()
            T_list.append(float(T_i))

        return np.array(T_list)


# ========================
# Sweep gap 
# ========================
gap_um_list = np.linspace(0.1, 1.0, 10)
gap_m_list = gap_um_list * 1e-6

T_results = []

for gap_um, gap_m in zip(gap_um_list, gap_m_list):
    print(f"Running gap = {gap_um:.3f} um")
    T_gap = run_single_gap(gap_m, save_fsp=True)
    T_results.append(T_gap)

T_results = np.array(T_results)   
T_plot = T_results.T              

# ========================
# Print results
# ========================
for i in range(NUM_OUTPUT_WG):
    print(f"\nDetector mon_out_{i:02d}")
    for g, t in zip(gap_um_list, T_plot[i, :]):
        print(f"gap = {g:.3f} um   T = {t:.6f}")

# ========================
# Plot results
# ========================
plt.figure(figsize=(6, 4))
for i in range(NUM_OUTPUT_WG):
    plt.plot(gap_um_list, T_plot[i, :], marker='o', label=f"det {i}")

plt.xlabel("Output waveguide gap (um)")
plt.ylabel("Transmission")
plt.title("Transmission vs output waveguide gap")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()